In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load files (they are in current working directory)
df_metric = pd.read_csv('metric.csv')
df_trace = pd.read_csv('trace.csv')
df_log = pd.read_csv('log.csv')
df_err = pd.read_csv('error_logs.csv')

# Helper to convert epoch seconds to Asia/Shanghai string
def ts_range(df, col='timestamp'):
    if df.empty or col not in df.columns:
        return (None, None)
    mn = int(df[col].min())
    mx = int(df[col].max())
    mn_str = pd.to_datetime(mn, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    mx_str = pd.to_datetime(mx, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    return mn_str, mx_str

# Compact top-N helper returning list of small dicts
def top_counts(df, col, n=20):
    if col not in df.columns:
        return []
    vc = df[col].value_counts().head(n)
    return [{'value': k, 'count': int(v)} for k, v in vc.items()]

# Summaries
metric_summary = {
    'rows': int(df_metric.shape[0]),
    'min_timestamp': ts_range(df_metric)[0],
    'max_timestamp': ts_range(df_metric)[1],
    'top_20_cmdb_id_counts': top_counts(df_metric, 'cmdb_id', 20),
    'unique_kpi_count': int(df_metric['kpi_name'].nunique()) if 'kpi_name' in df_metric.columns else 0,
    'unique_kpi_sample_top20': list(df_metric['kpi_name'].dropna().unique()[:20]) if 'kpi_name' in df_metric.columns else [],
    'top_20_kpi_name_counts': top_counts(df_metric, 'kpi_name', 20)
}

trace_summary = {
    'rows': int(df_trace.shape[0]),
    'min_timestamp': ts_range(df_trace)[0],
    'max_timestamp': ts_range(df_trace)[1],
    'top_20_cmdb_id_counts': top_counts(df_trace, 'cmdb_id', 20),
    'unique_trace_count': int(df_trace['trace_name'].nunique()) if 'trace_name' in df_trace.columns else 0,
    'unique_trace_sample_top20': list(df_trace['trace_name'].dropna().unique()[:20]) if 'trace_name' in df_trace.columns else [],
    'top_20_trace_name_counts': top_counts(df_trace, 'trace_name', 20)
}

log_summary = {
    'rows': int(df_log.shape[0]),
    'min_timestamp': ts_range(df_log)[0],
    'max_timestamp': ts_range(df_log)[1],
    'top_20_cmdb_id_counts': top_counts(df_log, 'cmdb_id', 20),
    'unique_log_count': int(df_log['log_name'].nunique()) if 'log_name' in df_log.columns else 0,
    'unique_log_sample_top20': list(df_log['log_name'].dropna().unique()[:20]) if 'log_name' in df_log.columns else [],
    'top_20_log_name_counts': top_counts(df_log, 'log_name', 20)
}

err_messages_top20 = []
if 'message' in df_err.columns:
    msg_vc = df_err['message'].fillna('').value_counts().head(20)
    err_messages_top20 = [{'message_sample': m, 'count': int(c)} for m, c in msg_vc.items()]

error_logs_summary = {
    'rows': int(df_err.shape[0]),
    'min_timestamp': ts_range(df_err)[0],
    'max_timestamp': ts_range(df_err)[1],
    'top_20_message_samples_with_counts': err_messages_top20,
    'top_20_cmdb_id_counts': top_counts(df_err, 'cmdb_id', 20)
}

summary = {
    'metric.csv': metric_summary,
    'trace.csv': trace_summary,
    'log.csv': log_summary,
    'error_logs.csv': error_logs_summary
}

summary

```
Out[1]:
```
```python
# Use the existing `summary` variable from previous execution to produce a plain-English summary.
s = summary  # reuse variable from kernel state

metric = s['metric.csv']
trace = s['trace.csv']
log = s['log.csv']
err = s['error_logs.csv']

summary_text = (
    "Summary of telemetry files:\n\n"
    f"metric.csv: {metric['rows']:,} rows covering {metric['min_timestamp']} to {metric['max_timestamp']}. "
    f"Top components by row count (top 5): {', '.join(f\"{item['value']} ({item['count']:,})\" for item in metric['top_20_cmdb_id_counts'][:5])}. "
    f"There are {metric['unique_kpi_count']:,} unique KPI names; the most frequent KPI series are container.node-6.* (network, memory, fs metrics) as shown in the top KPI counts.\n\n"
    f"trace.csv: {trace['rows']:,} rows covering {trace['min_timestamp']} to {trace['max_timestamp']}. "
    f"Top components by row count (top 5): {', '.join(f\"{item['value']} ({item['count']:,})\" for item in trace['top_20_cmdb_id_counts'][:5])}. "
    f"There are {trace['unique_trace_count']:,} unique trace names; the highest-frequency traces are from frontend pods (trace.from_frontend-1.*, -2.*, -0.*) and trace.self.*.\n\n"
    f"log.csv: {log['rows']:,} rows covering {log['min_timestamp']} to {log['max_timestamp']}. "
    f"Top components (partial): {', '.join(f\"{item['value']} ({item['count']:,})\" for item in log['top_20_cmdb_id_counts'][:5])}. "
    f"Only {log['unique_log_count']} unique log names were found: {', '.join(log['unique_log_sample_top20'])}. "
    f"The two log names appear equally often (about {log['top_20_log_name_counts'][0]['count']:,} each).\n\n"
    "error_logs.csv: 0 rows — no error log messages available.\n\n"
    "Overall takeaway: metric data is large and dominated by container.node-6 metrics and several services (adservice, adservice2, frontend pods). "
    "Trace data is concentrated on frontend and checkoutservice pods. Logs are sparse with just two normalized log metrics. No raw error log messages are present."
)

summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

{'metric.csv': {'rows': 319703, 'min_timestamp': '2022-03-20 16:30:00 CST', 'max_timestamp': '2022-03-20 17:30:00 CST', 'top_20_cmdb_id_counts': [{'value': 'adservice', 'count': 20801}, {'value': 'adservice2', 'count': 20313}, {'value': 'frontend2-0', 'count': 11956}, {'value': 'frontend-2', 'count': 11468}, {'value': 'frontend-1', 'count': 11468}, {'value': 'frontend-0', 'count': 11468}, {'value': 'checkoutservice-1', 'count': 8540}, {'value': 'checkoutservice2-0', 'count': 8540}, {'value': 'checkoutservice-2', 'count': 8296}, {'value': 'checkoutservice-0', 'count': 8296}, {'value': 'productcatalogservice-1', 'count': 7320}, {'value': 'productcatalogservice-0', 'count': 7320}, {'value': 'productcatalogservice-2', 'count': 7076}, {'value': 'productcatalogservice2-0', 'count': 6344}, {'value': 'recommendationservice2-0', 'count': 6222}, {'value': 'cartservice-2', 'count': 6100}, {'value': 'cartservice-0', 'count': 6100}, {'value': 'cartservice2-0', 'count': 6100}, {'value': 'recommendationservice-1', 'count': 5978}, {'value': 'recommendationservice-2', 'count': 5978}], 'unique_kpi_count': 1368, 'unique_kpi_sample_top20': ['app.grpc.count', 'app.grpc.mrt', 'app.grpc.rr', 'app.grpc.sr', 'app.http.count', 'app.http.mrt', 'app.http.rr', 'app.http.sr', 'runtime.java_lang_ClassLoading_LoadedClassCount', 'runtime.java_lang_ClassLoading_TotalLoadedClassCount', 'runtime.java_lang_ClassLoading_UnloadedClassCount', 'runtime.java_lang_ClassLoading_Verbose', 'runtime.java_lang_Compilation_CompilationTimeMonitoringSupported', 'runtime.java_lang_Compilation_TotalCompilationTime', 'runtime.java_lang_GarbageCollector_CollectionCount.Copy', 'runtime.java_lang_GarbageCollector_CollectionCount.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_CollectionTime.Copy', 'runtime.java_lang_GarbageCollector_CollectionTime.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_GcThreadCount.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_GcThreadCount.MarkSweepCompact'], 'top_20_kpi_name_counts': [{'value': 'container.node-6.container_network_receive_packets_dropped.eth0', 'count': 2196}, {'value': 'container.node-6.container_network_transmit_packets_dropped.eth0', 'count': 2196}, {'value': 'container.node-6.container_network_receive_MB.eth0', 'count': 2196}, {'value': 'container.node-6.container_network_receive_packets.eth0', 'count': 2196}, {'value': 'container.node-6.container_network_receive_errors.eth0', 'count': 2196}, {'value': 'container.node-6.container_network_transmit_errors.eth0', 'count': 2196}, {'value': 'container.node-6.container_network_transmit_packets.eth0', 'count': 2196}, {'value': 'container.node-6.container_network_transmit_MB.eth0', 'count': 2196}, {'value': 'container.node-6.container_memory_swap', 'count': 2074}, {'value': 'container.node-6.container_fs_reads./dev/vda1', 'count': 2074}, {'value': 'container.node-6.container_memory_working_set_MB', 'count': 2074}, {'value': 'container.node-6.container_fs_inodes./dev/vda1', 'count': 2074}, {'value': 'container.node-6.container_file_descriptors', 'count': 2074}, {'value': 'container.node-6.container_memory_max_usage_MB', 'count': 2074}, {'value': 'container.node-6.container_memory_rss', 'count': 2074}, {'value': 'container.node-6.container_memory_usage_MB', 'count': 2074}, {'value': 'container.node-6.container_tasks_state.sleeping', 'count': 2074}, {'value': 'container.node-6.container_tasks_state.running', 'count': 2074}, {'value': 'container.node-6.container_tasks_state.iowaiting', 'count': 2074}, {'value': 'container.node-6.container_fs_sector_reads./dev/vda1', 'count': 2074}]}, 'trace.csv': {'rows': 33696, 'min_timestamp': '2022-03-20 17:00:00 CST', 'max_timestamp': '2022-03-20 17:29:00 CST', 'top_20_cmdb_id_counts': [{'value': 'frontend-1', 'count': 2728}, {'value': 'frontend-2', 'count': 2596}, {'value': 'frontend-0', 'count': 2496}, {'value': 'checkoutservice-0', 'count': 2260}, {'value': 'checkoutservice-2', 'count': 2212}, {'value': 'checkoutservice-1', 'count': 2196}, {'value': 'productcatalogservice-1', 'count': 1164}, {'value': 'productcatalogservice-0', 'count': 1152}, {'value': 'productcatalogservice-2', 'count': 1140}, {'value': 'frontend2-0', 'count': 1080}, {'value': 'checkoutservice2-0', 'count': 1080}, {'value': 'recommendationservice-0', 'count': 840}, {'value': 'recommendationservice-2', 'count': 840}, {'value': 'recommendationservice-1', 'count': 840}, {'value': 'cartservice-1', 'count': 796}, {'value': 'cartservice-2', 'count': 796}, {'value': 'cartservice-0', 'count': 792}, {'value': 'currencyservice-0', 'count': 712}, {'value': 'currencyservice-2', 'count': 704}, {'value': 'currencyservice-1', 'count': 704}], 'unique_trace_count': 200, 'unique_trace_sample_top20': ['trace.from_frontend-0.duration_mean', 'trace.from_frontend-0.duration_p95', 'trace.from_frontend-0.error_rate', 'trace.from_frontend-0.row_count', 'trace.from_frontend-1.duration_mean', 'trace.from_frontend-1.duration_p95', 'trace.from_frontend-1.error_rate', 'trace.from_frontend-1.row_count', 'trace.from_frontend-2.duration_mean', 'trace.from_frontend-2.duration_p95', 'trace.from_frontend-2.error_rate', 'trace.from_frontend-2.row_count', 'trace.from_frontend2-0.duration_mean', 'trace.from_frontend2-0.duration_p95', 'trace.from_frontend2-0.error_rate', 'trace.from_frontend2-0.row_count', 'trace.from_checkoutservice-1.duration_mean', 'trace.from_checkoutservice-1.duration_p95', 'trace.from_checkoutservice-1.error_rate', 'trace.from_checkoutservice-1.row_count'], 'top_20_trace_name_counts': [{'value': 'trace.from_frontend-1.row_count', 'count': 622}, {'value': 'trace.from_frontend-1.error_rate', 'count': 622}, {'value': 'trace.from_frontend-1.duration_p95', 'count': 622}, {'value': 'trace.from_frontend-1.duration_mean', 'count': 622}, {'value': 'trace.from_frontend-2.row_count', 'count': 589}, {'value': 'trace.from_frontend-2.error_rate', 'count': 589}, {'value': 'trace.from_frontend-2.duration_p95', 'count': 589}, {'value': 'trace.from_frontend-2.duration_mean', 'count': 589}, {'value': 'trace.from_frontend-0.duration_mean', 'count': 564}, {'value': 'trace.from_frontend-0.duration_p95', 'count': 564}, {'value': 'trace.from_frontend-0.error_rate', 'count': 564}, {'value': 'trace.from_frontend-0.row_count', 'count': 564}, {'value': 'trace.self.row_count', 'count': 478}, {'value': 'trace.self.error_rate', 'count': 478}, {'value': 'trace.self.duration_p95', 'count': 478}, {'value': 'trace.self.duration_mean', 'count': 478}, {'value': 'trace.from_checkoutservice-0.row_count', 'count': 452}, {'value': 'trace.from_checkoutservice-0.error_rate', 'count': 452}, {'value': 'trace.from_checkoutservice-0.duration_p95', 'count': 452}, {'value': 'trace.from_checkoutservice-0.duration_mean', 'count': 452}]}, 'log.csv': {'rows': 1826, 'min_timestamp': '2022-03-20 17:00:00 CST', 'max_timestamp': '2022-03-20 17:30:00 CST', 'top_20_cmdb_id_counts': [{'value': 'cartservice-2', 'count': 62}, {'value': 'cartservice-1', 'count': 62}, {'value': 'cartservice-0', 'count': 62}, {'value': 'currencyservice-0', 'count': 62}, {'value': 'currencyservice-2', 'count': 62}, {'value': 'currencyservice-1', 'count': 62}, {'value': 'frontend-1', 'count': 62}, {'value': 'productcatalogservice-1', 'count': 62}, {'value': 'recommendationservice-2', 'count': 62}, {'value': 'shippingservice-1', 'count': 62}, {'value': 'recommendationservice-1', 'count': 62}, {'value': 'frontend-2', 'count': 62}, {'value': 'productcatalogservice-0', 'count': 62}, {'value': 'checkoutservice-0', 'count': 60}, {'value': 'adservice-2', 'count': 60}, {'value': 'productcatalogservice-2', 'count': 60}, {'value': 'shippingservice-0', 'count': 60}, {'value': 'emailservice-0', 'count': 60}, {'value': 'emailservice-2', 'count': 60}, {'value': 'checkoutservice-1', 'count': 60}], 'unique_log_count': 2, 'unique_log_sample_top20': ['log.error_count', 'log.row_count'], 'top_20_log_name_counts': [{'value': 'log.error_count', 'count': 913}, {'value': 'log.row_count', 'count': 913}]}, 'error_logs.csv': {'rows': 0, 'min_timestamp': None, 'max_timestamp': None, 'top_20_message_samples_with_counts': [], 'top_20_cmdb_id_counts': []}}```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
import pytz

# reuse df_metric from previous state if available, otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

# Ensure value is numeric
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

# Top 30 cmdb_id by total row count (global over full file)
top30_cmdb = df_metric['cmdb_id'].value_counts().head(30).index.tolist()

# Filter to those top 30 cmdb_ids
df_top30 = df_metric[df_metric['cmdb_id'].isin(top30_cmdb)].copy()

# Group by (cmdb_id, kpi_name) and compute required aggregates using full series
grouped = df_top30.groupby(['cmdb_id', 'kpi_name'])['value']

agg = grouped.agg(
    count='count',
    mean='mean',
    max='max',
    p95=lambda x: x.quantile(0.95),
    p5=lambda x: x.quantile(0.05)
).reset_index()

# Round numeric columns for compactness
agg[['mean', 'max', 'p95', 'p5']] = agg[['mean', 'max', 'p95', 'p5']].round(6)

# Sort by count desc and limit to top 50 rows
agg_top50 = agg.sort_values('count', ascending=False).head(50).reset_index(drop=True)

# Display the compact table
agg_top50

```
Out[2]:
```
```python
# Build a plain-English summary string from the existing aggregated results (agg_top50)
try:
    agg_top50
except NameError:
    # If agg_top50 not in state, reload minimal aggregation (fallback)
    import pandas as pd
    df_metric = pd.read_csv('metric.csv')
    df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
    top30_cmdb = df_metric['cmdb_id'].value_counts().head(30).index.tolist()
    df_top30 = df_metric[df_metric['cmdb_id'].isin(top30_cmdb)].copy()
    grouped = df_top30.groupby(['cmdb_id', 'kpi_name'])['value']
    agg = grouped.agg(count='count', mean='mean', max='max',
                      p95=lambda x: x.quantile(0.95),
                      p5=lambda x: x.quantile(0.05)).reset_index()
    agg[['mean','max','p95','p5']] = agg[['mean','max','p95','p5']].round(6)
    agg_top50 = agg.sort_values('count', ascending=False).head(50).reset_index(drop=True)

# summary computations
total_rows = int(agg_top50.shape[0])
distinct_counts = sorted(agg_top50['count'].unique())
top_row = agg_top50.iloc[0].to_dict()
examples = agg_top50.head(5)[['cmdb_id','kpi_name','count','mean','max','p95','p5']].to_dict(orient='records')

# detect presence of sentinel or extreme values
has_negative_one = (agg_top50[['mean','max','p95','p5']] == -1.0).any().any()
max_mean = float(agg_top50['mean'].max())
large_gc_present = any('GarbageCollector' in str(k) for k in agg_top50['kpi_name'].values)

summary_text = (
    f"Aggregates computed for top 30 components (restricted) and top {total_rows} (cmdb_id, kpi_name) pairs, "
    f"sorted by point count.\n\n"
    f"Key facts:\n"
    f"- Rows returned: {total_rows} pairs.\n"
    f"- Count values present in this table (distinct): {distinct_counts} (most rows show full coverage per-minute samples).\n"
    f"- Top pair (highest count): {top_row['cmdb_id']} | {top_row['kpi_name']} — count={int(top_row['count'])}, "
    f"mean={top_row['mean']}, max={top_row['max']}, p95={top_row['p95']}, p5={top_row['p5']}.\n\n"
    f"Example top entries (up to 5):\n"
    + "\n".join(f"  {i+1}. {r['cmdb_id']} | {r['kpi_name']} — count={int(r['count'])}, mean={r['mean']}, "
                f"max={r['max']}, p95={r['p95']}, p5={r['p5']}" for i,r in enumerate(examples))
    + "\n\n"
    f"Notable observations:\n"
    f"- Many pairs have identical high counts (e.g., 61), indicating full per-minute coverage across the time range for those metrics.\n"
    f"- High-volume application metrics appear (example: shippingservice jaeger/mesh metric has mean ~{examples[0]['mean'] if examples else 'N/A'}).\n"
    f"- adservice contains many KPI rows (e.g., app.grpc.count mean ~{next((r['mean'] for r in examples if r['cmdb_id']=='adservice'), 'N/A')}).\n"
    f"- Some KPI series include very large numeric GC fields and a few sentinel values (e.g., -1.0) were observed in the table.\n"
    f"- Overall: the top pairs capture per-component KPI distributions (count, mean, max, p95, p5); use this table to spot high averages, high maxima, "
    f"or unusual sentinel values for further root-cause inspection."
)

summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  count          mean           max           p95            p5
0   shippingservice-0  mesh.source.shippingservice.jaeger-collector.i...     61  1.139357e+03  2.238750e+03  1.542250e+03  5.970000e+02
1           adservice                                     app.grpc.count     61  2.483279e+02  2.990000e+02  2.900000e+02  1.850000e+02
2           adservice                                       app.grpc.mrt     61  2.550000e-03  2.670000e-03  2.644000e-03  2.453000e-03
3           adservice                                        app.grpc.rr     61  1.000000e+02  1.000000e+02  1.000000e+02  1.000000e+02
4           adservice                                        app.grpc.sr     61  1.000000e+02  1.000000e+02  1.000000e+02  1.000000e+02
..                ...                                                ...    ...           ...           ...           ...           ...
45          adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...     61  3.145411e+07  3.145411e+07  3.145411e+07  3.145411e+07
46          adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...     61  4.194304e+07  4.194304e+07  4.194304e+07  4.194304e+07
47          adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...     61  4.194304e+07  4.194304e+07  4.194304e+07  4.194304e+07
48          adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...     61 -1.000000e+00 -1.000000e+00 -1.000000e+00 -1.000000e+00
49          adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...     61 -1.000000e+00 -1.000000e+00 -1.000000e+00 -1.000000e+00

[50 rows x 7 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing variables if present
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

try:
    agg_top50
except NameError:
    # Recompute agg_top50 minimally (shouldn't usually happen)
    df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
    top30_cmdb = df_metric['cmdb_id'].value_counts().head(30).index.tolist()
    df_top30 = df_metric[df_metric['cmdb_id'].isin(top30_cmdb)].copy()
    grouped = df_top30.groupby(['cmdb_id', 'kpi_name'])['value']
    agg = grouped.agg(count='count', mean='mean', max='max',
                      p95=lambda x: x.quantile(0.95),
                      p5=lambda x: x.quantile(0.05)).reset_index()
    agg[['mean','max','p95','p5']] = agg[['mean','max','p95','p5']].round(6)
    agg_top50 = agg.sort_values('count', ascending=False).head(50).reset_index(drop=True)

# Ensure numeric
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
df_metric['timestamp'] = pd.to_numeric(df_metric['timestamp'], errors='coerce').astype('Int64')

# Incident window in Asia/Shanghai (UTC+8)
start_local = pd.Timestamp('2022-03-20 17:00:00').tz_localize(tz)
end_local = pd.Timestamp('2022-03-20 17:30:00').tz_localize(tz)
# Convert to UTC epoch seconds for comparison with metric timestamps (which are epoch seconds)
start_ts = int(start_local.tz_convert('UTC').timestamp())
end_ts = int(end_local.tz_convert('UTC').timestamp())

# Merge metric rows with agg_top50 thresholds for only those pairs present in agg_top50
pairs_df = agg_top50[['cmdb_id', 'kpi_name', 'p95', 'p5']].copy()
# ensure types align
df_merge = pd.merge(df_metric, pairs_df, on=['cmdb_id', 'kpi_name'], how='inner')

# Filter to incident window (inclusive)
df_window = df_merge[(df_merge['timestamp'] >= start_ts) & (df_merge['timestamp'] <= end_ts)].copy()

if df_window.empty:
    anomaly_list = []
else:
    # Determine anomaly flags using global thresholds from agg_top50 (p95/p5)
    df_window['is_high'] = df_window['value'] > df_window['p95']
    df_window['is_low'] = df_window['value'] < df_window['p5']
    df_window['is_anom'] = df_window['is_high'] | df_window['is_low']

    # Keep only anomaly rows for grouping stats
    anom_rows = df_window[df_window['is_anom']].copy()

    if anom_rows.empty:
        anomaly_list = []
    else:
        # Aggregations per pair
        def earliest_anom_ts(x):
            return int(x.min())

        agg_funcs = {
            'is_anom': 'sum',
            'is_high': 'sum',
            'is_low': 'sum',
            'value': ['max', 'min'],
            'timestamp': earliest_anom_ts,
            'p95': 'first',
            'p5': 'first'
        }
        g = anom_rows.groupby(['cmdb_id', 'kpi_name']).agg(agg_funcs)
        # flatten columns
        g.columns = ['anomaly_count_in_window', 'count_high', 'count_low',
                     'max_value_in_window', 'min_value_in_window',
                     'earliest_anomaly_timestamp', 'p95', 'p5']
        g = g.reset_index()

        # Determine direction and severity
        def compute_direction_severity(row):
            if row['count_high'] >= row['count_low']:
                direction = 'high'
                # avoid division by zero
                if row['p95'] == 0:
                    severity = np.inf
                else:
                    severity = row['max_value_in_window'] / row['p95']
            else:
                direction = 'low'
                # avoid division by zero
                if row['min_value_in_window'] == 0:
                    severity = np.inf
                else:
                    severity = row['p5'] / row['min_value_in_window']
            return pd.Series([direction, float(np.round(severity, 6))])

        g[['direction', 'severity']] = g.apply(compute_direction_severity, axis=1)

        # Format earliest timestamp to Asia/Shanghai string
        def fmt_ts(ts):
            try:
                return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
            except Exception:
                return None

        g['earliest_anomaly_timestamp'] = g['earliest_anomaly_timestamp'].apply(fmt_ts)

        # Select and sort top 20 by anomaly_count_in_window desc
        result_df = g[['cmdb_id', 'kpi_name', 'earliest_anomaly_timestamp',
                       'anomaly_count_in_window', 'direction', 'severity']].sort_values(
            'anomaly_count_in_window', ascending=False).head(20).reset_index(drop=True)

        # Convert to list of dicts as requested
        anomaly_list = result_df.to_dict(orient='records')

# Final compact result
anomaly_list

```
Out[3]:
```
```python
# Summarize anomaly_list from kernel state into a plain-English string
anoms = anomaly_list  # reuse existing variable

total_pairs = len(anoms)
high_count = sum(1 for a in anoms if a['direction'] == 'high')
low_count = sum(1 for a in anoms if a['direction'] == 'low')

top_examples = anoms[:5]  # already sorted by anomaly_count_in_window desc

lines = []
lines.append(f"Anomaly summary for 2022-03-20 17:00:00 to 17:30:00 CST:")
lines.append(f"- Total (cmdb_id, kpi_name) pairs with anomalies: {total_pairs}")
lines.append(f"- Directions: {high_count} high, {low_count} low")
lines.append("")
lines.append("Top anomalous pairs (by anomaly count):")
for i, e in enumerate(top_examples, 1):
    lines.append(
        f" {i}. {e['cmdb_id']} | {e['kpi_name']} — "
        f"earliest: {e['earliest_anomaly_timestamp']}, "
        f"count: {e['anomaly_count_in_window']}, "
        f"direction: {e['direction']}, "
        f"severity: {e['severity']}"
    )

# Notable observations
lines.append("")
lines.append("Notable observations:")
lines.append("- adservice.app.grpc.mrt had the most anomalies (5 points, high) starting at 17:00; severity ~1.01.")
lines.append("- shippingservice-0 shows multiple anomalies across container/network and memory KPIs (several high and low).")
lines.append("- One shippingservice metric (container_network_receive_MB) shows a large high severity (~7.72), indicating values far above its p95.")
lines.append("- Most anomalies are 'high' deviations. Use these pairs to prioritize further investigation (network and memory metrics on shippingservice-0, and several adservice RPC metrics).")

summary_text = "\n".join(lines)
summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

[{'cmdb_id': 'adservice', 'kpi_name': 'app.grpc.mrt', 'earliest_anomaly_timestamp': '2022-03-20 17:00:00 CST', 'anomaly_count_in_window': 5, 'direction': 'high', 'severity': 1.009818}, {'cmdb_id': 'shippingservice-0', 'kpi_name': 'container.node-6.container_fs_usage_MB./dev/vda1', 'earliest_anomaly_timestamp': '2022-03-20 17:27:00 CST', 'anomaly_count_in_window': 4, 'direction': 'high', 'severity': 1.000321}, {'cmdb_id': 'shippingservice-0', 'kpi_name': 'container.node-6.container_memory_working_set_MB', 'earliest_anomaly_timestamp': '2022-03-20 17:00:00 CST', 'anomaly_count_in_window': 4, 'direction': 'low', 'severity': 1.00047}, {'cmdb_id': 'adservice', 'kpi_name': 'app.grpc.count', 'earliest_anomaly_timestamp': '2022-03-20 17:15:00 CST', 'anomaly_count_in_window': 3, 'direction': 'high', 'severity': 1.031034}, {'cmdb_id': 'shippingservice-0', 'kpi_name': 'container.node-6.container_last_seen', 'earliest_anomaly_timestamp': '2022-03-20 17:28:00 CST', 'anomaly_count_in_window': 3, 'direction': 'high', 'severity': 1.0}, {'cmdb_id': 'shippingservice-0', 'kpi_name': 'container.node-6.container_memory_usage_MB', 'earliest_anomaly_timestamp': '2022-03-20 17:04:00 CST', 'anomaly_count_in_window': 3, 'direction': 'low', 'severity': 1.000385}, {'cmdb_id': 'shippingservice-0', 'kpi_name': 'container.node-6.container_network_receive_MB.eth0', 'earliest_anomaly_timestamp': '2022-03-20 17:07:00 CST', 'anomaly_count_in_window': 3, 'direction': 'high', 'severity': 7.720106}, {'cmdb_id': 'shippingservice-0', 'kpi_name': 'container.node-6.container_network_receive_packets.eth0', 'earliest_anomaly_timestamp': '2022-03-20 17:07:00 CST', 'anomaly_count_in_window': 2, 'direction': 'high', 'severity': 1.042353}, {'cmdb_id': 'shippingservice-0', 'kpi_name': 'container.node-6.container_network_transmit_MB.eth0', 'earliest_anomaly_timestamp': '2022-03-20 17:07:00 CST', 'anomaly_count_in_window': 2, 'direction': 'high', 'severity': 1.162795}, {'cmdb_id': 'shippingservice-0', 'kpi_name': 'container.node-6.container_network_transmit_packets.eth0', 'earliest_anomaly_timestamp': '2022-03-20 17:07:00 CST', 'anomaly_count_in_window': 2, 'direction': 'high', 'severity': 1.117318}, {'cmdb_id': 'shippingservice-0', 'kpi_name': 'mesh.source.shippingservice.jaeger-collector.istio_response_bytes.http.202.', 'earliest_anomaly_timestamp': '2022-03-20 17:06:00 CST', 'anomaly_count_in_window': 1, 'direction': 'high', 'severity': 1.032258}]```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# reuse anomaly_list from kernel state
try:
    anomaly_list
except NameError:
    anomaly_list = []

if not anomaly_list:
    result_list = []
else:
    df_anom = pd.DataFrame(anomaly_list)

    # Parse earliest_anomaly_timestamp (remove trailing timezone label like ' CST' if present)
    df_anom['earliest_ts_str'] = df_anom['earliest_anomaly_timestamp'].astype(str).str.replace(' CST', '', regex=False)
    df_anom['earliest_ts'] = pd.to_datetime(df_anom['earliest_ts_str'], format='%Y-%m-%d %H:%M:%S', errors='coerce')
    df_anom['earliest_ts'] = df_anom['earliest_ts'].dt.tz_localize(tz)

    # Ensure numeric columns
    df_anom['anomaly_count_in_window'] = pd.to_numeric(df_anom['anomaly_count_in_window'], errors='coerce').fillna(0).astype(int)
    df_anom['severity'] = pd.to_numeric(df_anom['severity'], errors='coerce').fillna(0.0)

    # Aggregate by cmdb_id
    rows = []
    grouped = df_anom.groupby('cmdb_id')
    for cmdb, g in grouped:
        earliest = g['earliest_ts'].min()
        total_kpi = int(g['kpi_name'].nunique())
        total_points = int(g['anomaly_count_in_window'].sum())
        max_sev = float(g['severity'].max())
        high_cnt = int((g['direction'] == 'high').sum())
        low_cnt = int((g['direction'] == 'low').sum())
        dominant = 'high' if high_cnt >= low_cnt else 'low'
        rows.append({
            'cmdb_id': cmdb,
            'earliest_anomaly_timestamp': earliest.strftime('%Y-%m-%d %H:%M:%S %Z'),
            'total_kpi_pairs_with_anomaly': total_kpi,
            'total_anomaly_points': total_points,
            'max_severity': round(max_sev, 6),
            'dominant_direction': dominant
        })

    result_df = pd.DataFrame(rows)
    result_df = result_df.sort_values('total_anomaly_points', ascending=False).head(20).reset_index(drop=True)
    result_list = result_df.to_dict(orient='records')

result_list

```
Out[4]:
```
```python
# Summarize the aggregated anomaly results (reuse result_list from kernel state)
try:
    result_list
except NameError:
    result_list = []

if not result_list:
    summary = "No anomalies detected in the provided anomaly_list."
else:
    # Build concise plain-English summary
    lines = ["Component-level anomaly summary (top components):"]
    for r in result_list:
        lines.append(
            f"- {r['cmdb_id']}: earliest anomaly {r['earliest_anomaly_timestamp']}, "
            f"{r['total_kpi_pairs_with_anomaly']} KPI pairs affected, "
            f"{r['total_anomaly_points']} total anomaly points, "
            f"max severity {r['max_severity']}, dominant direction {r['dominant_direction']}."
        )
    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

[{'cmdb_id': 'shippingservice-0', 'earliest_anomaly_timestamp': '2022-03-20 17:00:00 CST', 'total_kpi_pairs_with_anomaly': 9, 'total_anomaly_points': 24, 'max_severity': 7.720106, 'dominant_direction': 'high'}, {'cmdb_id': 'adservice', 'earliest_anomaly_timestamp': '2022-03-20 17:00:00 CST', 'total_kpi_pairs_with_anomaly': 2, 'total_anomaly_points': 8, 'max_severity': 1.031034, 'dominant_direction': 'high'}]```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz
import numpy as np

tz = pytz.timezone('Asia/Shanghai')

# Reuse df_trace if available, otherwise load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')

# Ensure correct types
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')
df_trace['timestamp'] = pd.to_numeric(df_trace['timestamp'], errors='coerce').astype('Int64')

# Group to compute count, mean, max, p95, p5
grouped = df_trace.groupby(['cmdb_id', 'trace_name'])['value']
agg_stats = grouped.agg(
    count='count',
    mean='mean',
    max='max',
    p95=lambda x: x.quantile(0.95),
    p5=lambda x: x.quantile(0.05)
).reset_index()

# Compute earliest timestamp where value == group max
group_max = agg_stats[['cmdb_id', 'trace_name', 'max']].rename(columns={'max': 'group_max'})
# Merge back to original rows
merged = pd.merge(df_trace, group_max, on=['cmdb_id', 'trace_name'], how='inner')
# Keep rows where value equals the group's max (use isclose for float robustness)
merged_max_rows = merged[np.isclose(merged['value'].fillna(0).astype(float), merged['group_max'].fillna(0).astype(float))]
# Get earliest timestamp of max per group
earliest_max_ts = merged_max_rows.groupby(['cmdb_id', 'trace_name'])['timestamp'].min().reset_index().rename(
    columns={'timestamp': 'earliest_timestamp_of_max_epoch'}
)

# Combine with agg_stats
result = pd.merge(agg_stats, earliest_max_ts, on=['cmdb_id', 'trace_name'], how='left')

# Format earliest timestamp to Asia/Shanghai string
def fmt_epoch_to_shanghai(e):
    try:
        return pd.to_datetime(int(e), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    except Exception:
        return None

result['earliest_timestamp_of_max'] = result['earliest_timestamp_of_max_epoch'].apply(fmt_epoch_to_shanghai)

# Round numeric columns for compactness
result[['mean', 'max', 'p95', 'p5']] = result[['mean', 'max', 'p95', 'p5']].round(6)

# Select desired columns and sort by max descending, limit to top 20
result_df = result[['cmdb_id', 'trace_name', 'count', 'mean', 'max', 'p95', 'p5', 'earliest_timestamp_of_max']].sort_values(
    by='max', ascending=False).head(20).reset_index(drop=True)

result_df

```
Out[5]:
```
```python
# Build a plain-English summary from the existing result_df in kernel state
try:
    result_df
except NameError:
    import pandas as pd
    result_df = pd.read_csv('trace.csv')  # fallback (not expected)

topN = result_df.head(20)
n_groups = topN.shape[0]

# Compose summary
lines = []
lines.append(f"Trace aggregation summary (top {n_groups} groups by max value):")
lines.append(f"- All groups use the full available trace data; most groups have count = 30 (full coverage).")

# Top 5 highlights
lines.append("\nTop 5 groups by peak (max) value:")
for i, row in topN.head(5).iterrows():
    lines.append(
        f"{i+1}. {row['cmdb_id']} | {row['trace_name']} — "
        f"count={int(row['count'])}, mean={row['mean']}, max={row['max']} (first at {row['earliest_timestamp_of_max']}), "
        f"p95={row['p95']}, p5={row['p5']}"
    )

# Observations
lines.append("\nNotable observations:")
lines.append("- The highest peak is frontend2-0 | trace.self.row_count (max=3107 at 2022-03-20 17:11 CST), with mean ~2828.")
lines.append("- Several high row_count peaks occur around 17:11 (also 17:13, 17:15, 17:22, 17:26, 17:29), indicating bursty request/load times.")
lines.append("- Many top groups are frontends and downstream services (productcatalog, cartservice, recommendationservice, adservice, currencyservice), showing traffic flow concentrations.")
lines.append("- Use these high-max groups to investigate request bursts, queueing, or downstream overload during those peak timestamps.")

summary_text = "\n".join(lines)
summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                   trace_name  count         mean     max      p95       p5 earliest_timestamp_of_max
0                frontend2-0                         trace.self.row_count     30  2827.666667  3107.0  2990.10  2676.25   2022-03-20 17:11:00 CST
1   productcatalogservice2-0             trace.from_frontend2-0.row_count     30  1304.433333  1512.0  1441.10  1193.90   2022-03-20 17:11:00 CST
2                frontend2-0  trace.to_productcatalogservice2-0.row_count     30  1304.433333  1512.0  1441.10  1193.90   2022-03-20 17:11:00 CST
3                 frontend-1                         trace.self.row_count     30   905.666667   999.0   992.20   818.45   2022-03-20 17:15:00 CST
4                frontend2-0        trace.to_currencyservice2-0.row_count     30   833.733333   963.0   917.00   736.35   2022-03-20 17:26:00 CST
5         currencyservice2-0             trace.from_frontend2-0.row_count     30   833.733333   963.0   917.00   736.35   2022-03-20 17:26:00 CST
6                 frontend-2                         trace.self.row_count     30   412.100000   571.0   508.35   344.80   2022-03-20 17:29:00 CST
7             cartservice2-0                         trace.self.row_count     30   293.933333   331.0   323.75   265.25   2022-03-20 17:11:00 CST
8                 frontend-0                         trace.self.row_count     30   256.533333   323.0   300.40   217.70   2022-03-20 17:11:00 CST
9                frontend2-0                    trace.from_root.row_count     30   283.000000   306.0   300.75   269.00   2022-03-20 17:11:00 CST
10               frontend2-0            trace.to_cartservice2-0.row_count     30   248.766667   281.0   270.20   228.90   2022-03-20 17:11:00 CST
11            cartservice2-0             trace.from_frontend2-0.row_count     30   248.766667   281.0   270.20   228.90   2022-03-20 17:11:00 CST
12  recommendationservice2-0                         trace.self.row_count     30   203.233333   237.0   224.00   186.00   2022-03-20 17:11:00 CST
13  recommendationservice2-0             trace.from_frontend2-0.row_count     30   203.233333   237.0   224.00   186.00   2022-03-20 17:11:00 CST
14               frontend2-0  trace.to_recommendationservice2-0.row_count     30   203.233333   237.0   224.00   186.00   2022-03-20 17:11:00 CST
15               frontend2-0              trace.to_adservice2-0.row_count     30   169.833333   188.0   185.00   153.45   2022-03-20 17:22:00 CST
16              adservice2-0             trace.from_frontend2-0.row_count     30   169.833333   188.0   185.00   153.45   2022-03-20 17:22:00 CST
17                frontend-1   trace.to_productcatalogservice-1.row_count     30   139.600000   161.0   158.75   124.45   2022-03-20 17:13:00 CST
18                frontend-1   trace.to_productcatalogservice-0.row_count     30   139.600000   161.0   158.20   123.90   2022-03-20 17:15:00 CST
19                frontend-1   trace.to_productcatalogservice-2.row_count     30   139.600000   161.0   158.30   124.45   2022-03-20 17:13:00 CST```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# reuse df_trace if exists, otherwise load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')

# Ensure correct types
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')
df_trace['timestamp'] = pd.to_numeric(df_trace['timestamp'], errors='coerce').astype('Int64')

# Compute global thresholds (p95 and p5) per (cmdb_id, trace_name) using full series (rule 9)
grouped_full = df_trace.groupby(['cmdb_id', 'trace_name'])['value']
thresholds = grouped_full.agg(
    p95=lambda x: float(x.quantile(0.95)),
    p5=lambda x: float(x.quantile(0.05)),
    count='count'
).reset_index()

# Restrict to trace_name containing any of the substrings
subs = ['duration', 'error_rate', 'row_count']
mask = thresholds['trace_name'].str.contains('|'.join(subs), case=False, na=False)
thr_filtered = thresholds[mask].copy()

# Incident window in Asia/Shanghai (UTC+8)
start_local = pd.Timestamp('2022-03-20 17:00:00').tz_localize(tz)
end_local = pd.Timestamp('2022-03-20 17:30:00').tz_localize(tz)
start_ts = int(start_local.tz_convert('UTC').timestamp())
end_ts = int(end_local.tz_convert('UTC').timestamp())

# Merge thresholds with trace data for only the groups of interest
df_interest = pd.merge(df_trace, thr_filtered[['cmdb_id', 'trace_name', 'p95', 'p5']], 
                       on=['cmdb_id', 'trace_name'], how='inner')

# Filter to incident window
df_window = df_interest[(df_interest['timestamp'] >= start_ts) & (df_interest['timestamp'] <= end_ts)].copy()

# If no rows in window -> empty result
if df_window.empty:
    anomalies_list = []
else:
    # Flag anomalies
    df_window['is_high'] = df_window['value'] > df_window['p95']
    df_window['is_low'] = df_window['value'] < df_window['p5']
    df_window['is_anom'] = df_window['is_high'] | df_window['is_low']

    anom_rows = df_window[df_window['is_anom']].copy()
    if anom_rows.empty:
        anomalies_list = []
    else:
        # Aggregate per group
        agg = anom_rows.groupby(['cmdb_id', 'trace_name']).agg(
            anomaly_count_in_window=('is_anom', 'sum'),
            count_high=('is_high', 'sum'),
            count_low=('is_low', 'sum'),
            earliest_anom_epoch=('timestamp', 'min'),
            max_value_in_window=('value', 'max'),
            min_value_in_window=('value', 'min'),
            p95=('p95', 'first'),
            p5=('p5', 'first')
        ).reset_index()

        # Determine direction and severity
        def dir_sev(row):
            if row['count_high'] >= row['count_low']:
                direction = 'high'
                if row['p95'] == 0:
                    severity = float('inf')
                else:
                    severity = row['max_value_in_window'] / row['p95']
            else:
                direction = 'low'
                if row['min_value_in_window'] == 0:
                    severity = float('inf')
                else:
                    severity = row['p5'] / row['min_value_in_window']
            return pd.Series([direction, float(np.round(severity, 6))])

        agg[['direction', 'severity']] = agg.apply(dir_sev, axis=1)

        # Format earliest timestamp to Asia/Shanghai
        def fmt_ts(e):
            try:
                return pd.to_datetime(int(e), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
            except Exception:
                return None

        agg['earliest_anomaly_timestamp'] = agg['earliest_anom_epoch'].apply(fmt_ts)

        # Select desired columns, sort by anomaly_count_in_window desc and limit to top 20
        result_df = agg[['cmdb_id', 'trace_name', 'earliest_anomaly_timestamp',
                         'anomaly_count_in_window', 'direction', 'severity']].sort_values(
            'anomaly_count_in_window', ascending=False).head(20).reset_index(drop=True)

        # Convert to list of dicts for return (compact)
        anomalies_list = result_df.to_dict(orient='records')

# Final compact output variable
anomalies_list

```
Out[6]:
```
```python
# Build a plain-English summary from the detected trace anomalies (reuse anomalies_list)
anoms = anomalies_list  # reuse kernel variable

# Compose summary
total_groups = len(anoms)
high_count = sum(1 for a in anoms if a['direction'] == 'high')
low_count = sum(1 for a in anoms if a['direction'] == 'low')

# Top components by number of anomaly groups
from collections import Counter
cmdb_counts = Counter(a['cmdb_id'] for a in anoms)
top_components = cmdb_counts.most_common(5)

# Top severe anomalies (by severity)
top_severe = sorted(anoms, key=lambda x: x['severity'] if x.get('severity') is not None else 0, reverse=True)[:5]

summary_lines = []
summary_lines.append(f"Trace anomaly summary for 2022-03-20 17:00–17:30 CST:")
summary_lines.append(f"- Total anomalous (cmdb_id, trace_name) groups detected: {total_groups} (high={high_count}, low={low_count}).")
if top_components:
    summary_lines.append("- Top affected components (by number of anomalous trace groups): " +
                         ", ".join(f"{c} ({n})" for c,n in top_components) + ".")
summary_lines.append("- Most anomalies are 'high' deviations (increased durations/error rates/row counts).")
summary_lines.append("- Earliest anomalies for many groups occur around 17:00–17:04; several bursts appear up to ~17:07 and later.")

if top_severe:
    summary_lines.append("- Top severe anomalous trace groups (examples):")
    for t in top_severe:
        summary_lines.append(
            f"  • {t['cmdb_id']} | {t['trace_name']} — earliest {t['earliest_anomaly_timestamp']}, "
            f"count={t['anomaly_count_in_window']}, direction={t['direction']}, severity={t['severity']}"
        )

summary_lines.append("")
summary_lines.append("Interpretation and next steps:")
summary_lines.append("- Frontend pods (notably frontend-1 and frontend-2) show multiple high-duration and elevated row_count anomalies, starting near 17:00–17:02.")
summary_lines.append("- Outbound calls from frontends to adservice and checkoutservice show some of the highest severity increases; prioritize investigating those service calls and their downstream services.")
summary_lines.append("- Focus inspection on request/latency spikes between 17:00 and 17:10, check service logs, resource usage, and network for the involved frontends and adservice/checkoutservice.")

summary_text = "\n".join(summary_lines)
summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

[{'cmdb_id': 'adservice-1', 'trace_name': 'trace.from_frontend-1.duration_mean', 'earliest_anomaly_timestamp': '2022-03-20 17:04:00 CST', 'anomaly_count_in_window': 4, 'direction': 'high', 'severity': 1.074169}, {'cmdb_id': 'frontend-1', 'trace_name': 'trace.to_productcatalogservice-2.row_count', 'earliest_anomaly_timestamp': '2022-03-20 17:01:00 CST', 'anomaly_count_in_window': 4, 'direction': 'high', 'severity': 1.017056}, {'cmdb_id': 'frontend-1', 'trace_name': 'trace.to_recommendationservice-0.duration_mean', 'earliest_anomaly_timestamp': '2022-03-20 17:01:00 CST', 'anomaly_count_in_window': 4, 'direction': 'high', 'severity': 1.021411}, {'cmdb_id': 'frontend-1', 'trace_name': 'trace.to_recommendationservice-0.duration_p95', 'earliest_anomaly_timestamp': '2022-03-20 17:01:00 CST', 'anomaly_count_in_window': 4, 'direction': 'high', 'severity': 1.027596}, {'cmdb_id': 'frontend-1', 'trace_name': 'trace.to_recommendationservice-1.duration_mean', 'earliest_anomaly_timestamp': '2022-03-20 17:01:00 CST', 'anomaly_count_in_window': 4, 'direction': 'high', 'severity': 1.070684}, {'cmdb_id': 'frontend-1', 'trace_name': 'trace.to_recommendationservice-1.duration_p95', 'earliest_anomaly_timestamp': '2022-03-20 17:02:00 CST', 'anomaly_count_in_window': 4, 'direction': 'high', 'severity': 1.031955}, {'cmdb_id': 'frontend-1', 'trace_name': 'trace.to_recommendationservice-2.duration_mean', 'earliest_anomaly_timestamp': '2022-03-20 17:06:00 CST', 'anomaly_count_in_window': 4, 'direction': 'high', 'severity': 1.003941}, {'cmdb_id': 'frontend-2', 'trace_name': 'trace.from_root.duration_p95', 'earliest_anomaly_timestamp': '2022-03-20 17:12:00 CST', 'anomaly_count_in_window': 4, 'direction': 'high', 'severity': 1.221123}, {'cmdb_id': 'frontend-2', 'trace_name': 'trace.self.duration_mean', 'earliest_anomaly_timestamp': '2022-03-20 17:06:00 CST', 'anomaly_count_in_window': 4, 'direction': 'high', 'severity': 1.091401}, {'cmdb_id': 'frontend-2', 'trace_name': 'trace.self.duration_p95', 'earliest_anomaly_timestamp': '2022-03-20 17:06:00 CST', 'anomaly_count_in_window': 4, 'direction': 'high', 'severity': 1.011276}, {'cmdb_id': 'frontend-1', 'trace_name': 'trace.to_recommendationservice-2.duration_p95', 'earliest_anomaly_timestamp': '2022-03-20 17:03:00 CST', 'anomaly_count_in_window': 4, 'direction': 'high', 'severity': 1.030766}, {'cmdb_id': 'frontend-2', 'trace_name': 'trace.self.row_count', 'earliest_anomaly_timestamp': '2022-03-20 17:00:00 CST', 'anomaly_count_in_window': 4, 'direction': 'high', 'severity': 1.123242}, {'cmdb_id': 'frontend-2', 'trace_name': 'trace.to_adservice-1.duration_p95', 'earliest_anomaly_timestamp': '2022-03-20 17:07:00 CST', 'anomaly_count_in_window': 4, 'direction': 'high', 'severity': 1.67264}, {'cmdb_id': 'frontend-1', 'trace_name': 'trace.to_shippingservice-2.duration_p95', 'earliest_anomaly_timestamp': '2022-03-20 17:07:00 CST', 'anomaly_count_in_window': 4, 'direction': 'high', 'severity': 1.030393}, {'cmdb_id': 'frontend-1', 'trace_name': 'trace.to_shippingservice-2.row_count', 'earliest_anomaly_timestamp': '2022-03-20 17:02:00 CST', 'anomaly_count_in_window': 4, 'direction': 'high', 'severity': 1.04712}, {'cmdb_id': 'frontend-2', 'trace_name': 'trace.from_root.duration_mean', 'earliest_anomaly_timestamp': '2022-03-20 17:01:00 CST', 'anomaly_count_in_window': 4, 'direction': 'high', 'severity': 1.118111}, {'cmdb_id': 'frontend-1', 'trace_name': 'trace.to_checkoutservice-2.duration_mean', 'earliest_anomaly_timestamp': '2022-03-20 17:02:00 CST', 'anomaly_count_in_window': 4, 'direction': 'high', 'severity': 1.214925}, {'cmdb_id': 'frontend-1', 'trace_name': 'trace.to_checkoutservice-2.duration_p95', 'earliest_anomaly_timestamp': '2022-03-20 17:02:00 CST', 'anomaly_count_in_window': 4, 'direction': 'high', 'severity': 1.038381}, {'cmdb_id': 'frontend-1', 'trace_name': 'trace.to_adservice-2.duration_p95', 'earliest_anomaly_timestamp': '2022-03-20 17:04:00 CST', 'anomaly_count_in_window': 4, 'direction': 'high', 'severity': 1.721664}, {'cmdb_id': 'frontend-1', 'trace_name': 'trace.to_cartservice-0.duration_mean', 'earliest_anomaly_timestamp': '2022-03-20 17:02:00 CST', 'anomaly_count_in_window': 4, 'direction': 'high', 'severity': 1.176803}]```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing dataframes if present
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

try:
    df_log
except NameError:
    df_log = pd.read_csv('log.csv')

# Ensure types
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
df_metric['timestamp'] = pd.to_numeric(df_metric['timestamp'], errors='coerce').astype('Int64')

df_log['value'] = pd.to_numeric(df_log['value'], errors='coerce')
df_log['timestamp'] = pd.to_numeric(df_log['timestamp'], errors='coerce').astype('Int64')

# Incident window (Asia/Shanghai) -> convert to UTC epoch seconds for filtering
start_local = pd.Timestamp('2022-03-20 17:00:00').tz_localize(tz)
end_local = pd.Timestamp('2022-03-20 17:30:00').tz_localize(tz)
start_ts = int(start_local.tz_convert('UTC').timestamp())
end_ts = int(end_local.tz_convert('UTC').timestamp())

# Define substring filters for requested components
ship_subs = [
    'container_network_receive_MB',
    'container_network_receive_packets',
    'container_network_transmit_MB',
    'container_network_transmit_packets',
    'container_memory_working_set_MB',
    'container_memory_usage_MB',
    'container_fs_usage_MB',
    'container_last_seen'
]
ad_subs = ['app.grpc.mrt', 'app.grpc.count']
# frontend requested kpi_name containing 'trace' or 'app.http'
frontend_subs = ['trace', 'app.http']

# Build boolean masks
mask_time = (df_metric['timestamp'] >= start_ts) & (df_metric['timestamp'] <= end_ts)

mask_ship = (df_metric['cmdb_id'] == 'shippingservice-0') & df_metric['kpi_name'].str.contains('|'.join(ship_subs), na=False)
mask_ad = (df_metric['cmdb_id'] == 'adservice') & df_metric['kpi_name'].str.contains('|'.join(ad_subs), na=False)
mask_front1 = (df_metric['cmdb_id'] == 'frontend-1') & df_metric['kpi_name'].str.contains('|'.join(frontend_subs), na=False, case=False)
mask_front2 = (df_metric['cmdb_id'] == 'frontend-2') & df_metric['kpi_name'].str.contains('|'.join(frontend_subs), na=False, case=False)

mask_any = mask_time & (mask_ship | mask_ad | mask_front1 | mask_front2)

df_matches = df_metric[mask_any].copy()

# Convert timestamp to Asia/Shanghai readable string for clarity in output
def fmt_sh(ts):
    try:
        return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    except Exception:
        return None

df_matches['timestamp_sh'] = df_matches['timestamp'].apply(fmt_sh)

# Select requested columns and sort by timestamp, limit to first 50 rows
df_metric_out = df_matches.sort_values('timestamp').loc[:, ['timestamp_sh', 'cmdb_id', 'kpi_name', 'value']].head(50)
df_metric_out = df_metric_out.rename(columns={'timestamp_sh': 'timestamp'})

# ---- Logs aggregation for specified cmdb_ids ----
cmdb_list = ['shippingservice-0', 'adservice', 'frontend-1', 'frontend-2', 'frontend-0']

mask_log_time = (df_log['timestamp'] >= start_ts) & (df_log['timestamp'] <= end_ts)
df_log_window = df_log[mask_log_time & df_log['cmdb_id'].isin(cmdb_list)].copy()

# Prepare aggregation
# Sum of values where log_name == 'log.error_count' and where 'log.row_count'
agg_rows = []
for cm in cmdb_list:
    dfc = df_log_window[df_log_window['cmdb_id'] == cm]
    sum_err = dfc.loc[dfc['log_name'] == 'log.error_count', 'value'].sum(min_count=1)
    sum_row = dfc.loc[dfc['log_name'] == 'log.row_count', 'value'].sum(min_count=1)
    row_cnt = int(dfc.shape[0])
    # convert possible NaN sums to 0
    sum_err = float(0 if np.isnan(sum_err) else sum_err)
    sum_row = float(0 if np.isnan(sum_row) else sum_row)
    agg_rows.append({
        'cmdb_id': cm,
        'sum_log_error_count': sum_err,
        'sum_log_row_count': sum_row,
        'row_counts': row_cnt
    })

df_log_agg = pd.DataFrame(agg_rows)

# Display compact results: df_metric_out (<=50 rows) and df_log_agg (one row per cmdb_id)
df_metric_out, df_log_agg

```
Out[7]:
```
Summary of requested extracts (2022-03-20 17:00:00 — 17:30:00 CST):

Metric rows (first 50 matches, sorted by timestamp)
- Results include adservice app.grpc.count and app.grpc.mrt at 17:00 and subsequent times, and many shippingservice-0 container metrics (e.g., container_fs_usage_MB ≈ 54.7 MB, container_last_seen epoch values, container_memory_usage_MB ≈ 50+ MB, and other container_network and memory KPIs) around 17:00–17:04.  
- The extract was limited to 50 rows and contains the requested columns: timestamp, cmdb_id, kpi_name, value. (These 50 rows show shippingservice-0 metrics repeated each minute and adservice grpc metrics at multiple timestamps.)

Log aggregates (sums over same window)
- shippingservice-0: sum_log_error_count = 0.0, sum_log_row_count = 1690.0, row_counts = 60
- adservice: sum_log_error_count = 0.0, sum_log_row_count = 0.0, row_counts = 0  (no log rows in window)
- frontend-1: sum_log_error_count = 0.0, sum_log_row_count = 41274.0, row_counts = 62
- frontend-2: sum_log_error_count = 0.0, sum_log_row_count = 19076.0, row_counts = 62
- frontend-0: sum_log_error_count = 0.0, sum_log_row_count = 11969.0, row_counts = 60

Notable points / quick interpretation
- shippingservice-0 metrics in the window show per-minute container resource and network measurements (useful to correlate with earlier anomalies).  
- adservice metrics (app.grpc.count and app.grpc.mrt) are present and can be used to corroborate RPC traffic/latency spikes.  
- Frontends have large log.row_count totals in the window (frontend-1 highest), while none of these components reported log.error_count > 0 in the logs for this window.  
- adservice has no log rows in log.csv for this interval (row_counts = 0), so rely on metrics/traces for its behavior.

If you want, I can (a) show the full 50 metric rows (table) or (b) filter metric rows to only show specific network or memory KPI names in more detail.

The original code execution output of IPython Kernel is also provided below for reference:

(                      timestamp            cmdb_id                                          kpi_name         value
157230  2022-03-20 17:00:00 CST          adservice                                    app.grpc.count  2.520000e+02
157231  2022-03-20 17:00:00 CST          adservice                                      app.grpc.mrt  2.669444e-03
162174  2022-03-20 17:00:00 CST  shippingservice-0  container.node-6.container_fs_usage_MB./dev/vda1  5.469336e+01
162180  2022-03-20 17:00:00 CST  shippingservice-0              container.node-6.container_last_seen  1.647767e+09
162191  2022-03-20 17:00:00 CST  shippingservice-0        container.node-6.container_memory_usage_MB  5.094922e+01
...                         ...                ...                                               ...           ...
172956  2022-03-20 17:03:00 CST          adservice                                      app.grpc.mrt  2.559804e-03
177898  2022-03-20 17:03:00 CST  shippingservice-0  container.node-6.container_fs_usage_MB./dev/vda1  5.470898e+01
177904  2022-03-20 17:03:00 CST  shippingservice-0              container.node-6.container_last_seen  1.647767e+09
183145  2022-03-20 17:04:00 CST  shippingservice-0              container.node-6.container_last_seen  1.647767e+09
183139  2022-03-20 17:04:00 CST  shippingservice-0  container.node-6.container_fs_usage_MB./dev/vda1  5.471484e+01

[50 rows x 4 columns],              cmdb_id  sum_log_error_count  sum_log_row_count  row_counts
0  shippingservice-0                  0.0             1690.0          60
1          adservice                  0.0                0.0           0
2         frontend-1                  0.0            41274.0          62
3         frontend-2                  0.0            19076.0          62
4         frontend-0                  0.0            11969.0          60)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing dataframes/variables from kernel if available
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

# Ensure numeric types
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
df_metric['timestamp'] = pd.to_numeric(df_metric['timestamp'], errors='coerce').astype('Int64')

# Reuse agg_top50 if present (contains global p95/p5 for many pairs)
try:
    agg_top50
except NameError:
    agg_top50 = None

# Define patterns
ship_subs = [
    'container_network_receive_MB',
    'container_network_receive_packets',
    'container_network_transmit_MB',
    'container_network_transmit_packets',
    'container_memory_working_set_MB',
    'container_memory_usage_MB',
    'container_fs_usage_MB',
    'container_last_seen'
]
ad_subs = ['app.grpc.mrt', 'app.grpc.count']

# Find all matching (cmdb_id, kpi_name) pairs in the full metric series (global)
mask_ship_pairs = (df_metric['cmdb_id'] == 'shippingservice-0') & df_metric['kpi_name'].str.contains('|'.join(ship_subs), na=False)
mask_ad_pairs = (df_metric['cmdb_id'] == 'adservice') & df_metric['kpi_name'].str.contains('|'.join(ad_subs), na=False)

pairs_df = df_metric[mask_ship_pairs | mask_ad_pairs][['cmdb_id', 'kpi_name']].drop_duplicates().reset_index(drop=True)

# Build thresholds: prefer agg_top50 values if present; otherwise compute global p95/p5 using full series (rule 9)
if agg_top50 is not None:
    thr_from_agg = agg_top50[['cmdb_id', 'kpi_name', 'p95', 'p5']].copy()
else:
    thr_from_agg = pd.DataFrame(columns=['cmdb_id', 'kpi_name', 'p95', 'p5'])

# Merge to see which pairs need computation
pairs_with_thr = pd.merge(pairs_df, thr_from_agg, on=['cmdb_id', 'kpi_name'], how='left')

# Compute missing thresholds using full df_metric (global)
missing = pairs_with_thr[pairs_with_thr['p95'].isna()][['cmdb_id', 'kpi_name']]
if not missing.empty:
    grp = df_metric.groupby(['cmdb_id', 'kpi_name'])['value']
    computed = grp.agg(p95=lambda x: float(x.quantile(0.95)), p5=lambda x: float(x.quantile(0.05))).reset_index()
    # Keep only needed pairs
    computed = pd.merge(missing, computed, on=['cmdb_id', 'kpi_name'], how='left')
else:
    computed = pd.DataFrame(columns=['cmdb_id', 'kpi_name', 'p95', 'p5'])

# Consolidate thresholds
thresholds = pd.concat([thr_from_agg, computed], ignore_index=True, sort=False)
# Keep only thresholds for our pairs
thresholds = pd.merge(pairs_df, thresholds, on=['cmdb_id', 'kpi_name'], how='left')

# Incident window (Asia/Shanghai) -> convert to UTC epoch seconds for filtering
start_local = pd.Timestamp('2022-03-20 17:00:00').tz_localize(tz)
end_local = pd.Timestamp('2022-03-20 17:30:00').tz_localize(tz)
start_ts = int(start_local.tz_convert('UTC').timestamp())
end_ts = int(end_local.tz_convert('UTC').timestamp())

# Filter df_metric to the window and to the pairs of interest
df_window = df_metric[(df_metric['timestamp'] >= start_ts) & (df_metric['timestamp'] <= end_ts)].copy()
df_window = pd.merge(df_window, thresholds, on=['cmdb_id', 'kpi_name'], how='inner')

# If no data, return empty list
if df_window.empty:
    segments_df = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','direction','segment_start_timestamp','segment_end_timestamp',
        'segment_length','max_value_in_segment','p95','p5','severity'
    ])
else:
    # Determine anomaly direction per row using global thresholds (p95/p5)
    df_window['is_high'] = df_window['value'] > df_window['p95']
    df_window['is_low'] = df_window['value'] < df_window['p5']
    # Keep only anomaly rows (high or low)
    df_anom = df_window[df_window['is_high'] | df_window['is_low']].copy()
    # Assign direction label per row
    df_anom['direction'] = np.where(df_anom['is_high'], 'high', 'low')

    if df_anom.empty:
        segments_df = pd.DataFrame(columns=[
            'cmdb_id','kpi_name','direction','segment_start_timestamp','segment_end_timestamp',
            'segment_length','max_value_in_segment','p95','p5','severity'
        ])
    else:
        # Sort to detect consecutive timestamps
        df_anom = df_anom.sort_values(['cmdb_id', 'kpi_name', 'direction', 'timestamp']).reset_index(drop=True)

        # Compute previous row context to identify new segments
        df_anom['prev_cmdb'] = df_anom['cmdb_id'].shift(1)
        df_anom['prev_kpi'] = df_anom['kpi_name'].shift(1)
        df_anom['prev_dir'] = df_anom['direction'].shift(1)
        df_anom['prev_ts'] = df_anom['timestamp'].shift(1)

        # A new segment starts when any of these conditions hold
        df_anom['new_segment'] = (
            (df_anom['cmdb_id'] != df_anom['prev_cmdb']) |
            (df_anom['kpi_name'] != df_anom['prev_kpi']) |
            (df_anom['direction'] != df_anom['prev_dir']) |
            ((df_anom['timestamp'] - df_anom['prev_ts']) != 60)
        )

        # First row is a new segment
        df_anom.loc[df_anom.index[0], 'new_segment'] = True

        # Segment id by cumulative sum
        df_anom['segment_id'] = df_anom['new_segment'].cumsum()

        # Aggregate per segment
        seg_grp = df_anom.groupby(['segment_id', 'cmdb_id', 'kpi_name', 'direction', 'p95', 'p5'])
        segs = seg_grp.agg(
            segment_start_epoch=('timestamp', 'min'),
            segment_end_epoch=('timestamp', 'max'),
            segment_length=('timestamp', 'count'),
            max_value_in_segment=('value', 'max'),
            min_value_in_segment=('value', 'min')
        ).reset_index()

        # Filter out isolated single-point spikes (keep length >=2)
        segs = segs[segs['segment_length'] >= 2].copy()

        if segs.empty:
            segments_df = pd.DataFrame(columns=[
                'cmdb_id','kpi_name','direction','segment_start_timestamp','segment_end_timestamp',
                'segment_length','max_value_in_segment','p95','p5','severity'
            ])
        else:
            # Compute severity
            def compute_sev(row):
                if row['direction'] == 'high':
                    if row['p95'] == 0:
                        return float('inf')
                    return float(row['max_value_in_segment'] / row['p95'])
                else:
                    if row['min_value_in_segment'] == 0:
                        return float('inf')
                    return float(row['p5'] / row['min_value_in_segment'])

            segs['severity'] = segs.apply(compute_sev, axis=1)

            # Format timestamps to Asia/Shanghai strings
            def fmt_ts(e):
                return pd.to_datetime(int(e), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

            segs['segment_start_timestamp'] = segs['segment_start_epoch'].apply(fmt_ts)
            segs['segment_end_timestamp'] = segs['segment_end_epoch'].apply(fmt_ts)

            # Select and order columns
            segments_df = segs[[
                'cmdb_id','kpi_name','direction','segment_start_timestamp','segment_end_timestamp',
                'segment_length','max_value_in_segment','p95','p5','severity'
            ]].copy()

            # Round numeric columns for compactness
            segments_df[['max_value_in_segment','p95','p5','severity']] = segments_df[[
                'max_value_in_segment','p95','p5','severity'
            ]].round(6)

            # Sort by segment_start_timestamp ascending and limit to top 20
            segments_df['segment_start_epoch'] = segs['segment_start_epoch']
            segments_df = segments_df.sort_values('segment_start_epoch', ascending=True).head(20).reset_index(drop=True)
            segments_df = segments_df.drop(columns=['segment_start_epoch'])

# Final compact output (top segments)
segments_df.head(20)

```
Out[8]:
```
```python
# Summarize the detected consecutive anomaly segments (reuse segments_df from kernel state)
try:
    segments_df
except NameError:
    segments_df = None

if segments_df is None or segments_df.empty:
    summary = "No multi-point consecutive anomaly segments (length>=2) found for the requested pairs in the 2022-03-20 17:00–17:30 CST window."
else:
    # Build concise summary
    lines = []
    lines.append("Consecutive anomaly segments (length >= 2) within 2022-03-20 17:00–17:30 CST:")
    lines.append(f"- Total segments found: {len(segments_df)}. Listed below are highlights sorted by start time.\n")

    # Iterate rows to create brief descriptions
    for idx, r in segments_df.iterrows():
        lines.append(
            f"{idx+1}. {r['cmdb_id']} | {r['kpi_name']} — {r['direction'].upper()} from {r['segment_start_timestamp']} "
            f"to {r['segment_end_timestamp']} ({int(r['segment_length'])} points); "
            f"max={r['max_value_in_segment']}, p95={r['p95']}, p5={r['p5']}, severity={r['severity']}"
        )

    # Notable interpretation
    lines.append("\nNotable findings and recommended focus:")
    lines.append("- shippingservice-0 shows several multi-point segments:")
    lines.append("  * Memory metrics (working_set_MB and memory_usage_MB) dipped low around 17:04–17:06 (3 consecutive points). Severity ~1.0004 (marginal).")
    lines.append("  * A pronounced network receive MB spike occurred at 17:25–17:26 (2 points) with very high severity (~7.72) — this is the most significant anomaly and should be prioritized.")
    lines.append("  * Filesystem usage (container_fs_usage_MB) shows a 4-point high segment from 17:27–17:30 (severity ~1.0003) — small relative increase but persistent for several minutes.")
    lines.append("  * container_last_seen shows a 3-point high segment at 17:28–17:30 (severity ~1.0) — likely last-seen heartbeat timestamps, less actionable by itself.")
    lines.append("- adservice has a 2-point high segment on app.grpc.mrt at 17:00–17:01 (severity ~1.0096) indicating a small but multi-point latency elevation early in the window.")
    lines.append("\nActionable priority:")
    lines.append("1) Immediately investigate the shippingservice-0 network receive MB spike at ~17:25–17:26 (highest severity). Correlate with traces, network metrics, and pod/node-level networking.")
    lines.append("2) Check persistent filesystem usage rise (17:27–17:30) and whether it impacted I/O or triggered throttling.")
    lines.append("3) Examine the 17:04–17:06 memory dips for shippingservice-0 to rule out memory reclaim/GC or reporting artifacts.")
    lines.append("4) Review adservice RPC latency around 17:00 for upstream impact on request latency/throughput.")
    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name direction  segment_start_timestamp    segment_end_timestamp  segment_length  max_value_in_segment           p95            p5  severity
0          adservice                                       app.grpc.mrt      high  2022-03-20 17:00:00 CST  2022-03-20 17:01:00 CST               2          2.669000e-03  2.644000e-03  2.453000e-03  1.009623
1  shippingservice-0         container.node-6.container_memory_usage_MB       low  2022-03-20 17:04:00 CST  2022-03-20 17:06:00 CST               3          5.076172e+01  5.094922e+01  5.076953e+01  1.000385
2  shippingservice-0   container.node-6.container_memory_working_set_MB       low  2022-03-20 17:04:00 CST  2022-03-20 17:06:00 CST               3          4.158594e+01  4.267578e+01  4.159375e+01  1.000470
3  shippingservice-0  container.node-6.container_network_receive_MB....      high  2022-03-20 17:25:00 CST  2022-03-20 17:26:00 CST               2          3.249390e-01  4.209000e-02  2.422700e-02  7.720106
4  shippingservice-0   container.node-6.container_fs_usage_MB./dev/vda1      high  2022-03-20 17:27:00 CST  2022-03-20 17:30:00 CST               4          5.485156e+01  5.483398e+01  5.457422e+01  1.000321
5  shippingservice-0               container.node-6.container_last_seen      high  2022-03-20 17:28:00 CST  2022-03-20 17:30:00 CST               3          1.647769e+09  1.647768e+09  1.647765e+09  1.000000```
```